# AniList Anime Dataset — Data Cleaning Script
> Nova IMS · 2025/2026 · Group 16

In [173]:
"""
Steps:
  1. Load CSV
  2. Drop useless columns (mb)
  3. Fix data types (if necessary)
  4. Basic data quality (nulls, duplicates)
  5. Parse JSON columns → Python objects (a simple flaw in dataset)
  6. Save cleaned output as JSON (ready for MongoDB import, prob not doing it today)
"""

import json
import pandas as pd
import os

---
# 1. Load

In [174]:
INPUT_PATH  = "../Data/anilist_anime_data_complete.csv"
OUTPUT_PATH = "../Data/anilist_cleaned.json"

print("Loading CSV...")
df = pd.read_csv(INPUT_PATH)
print(f"  Loaded: {len(df)} rows × {len(df.columns)} columns")

Loading CSV...
  Loaded: 20329 rows × 62 columns


---
# 2. Drop Useless columns
### What exactly dropped:

**'Unnamed: 0'** - pandas index<br>
**'type'**       - always "ANIME", no value<br>
**'isFavourite'** 'isLocked' - user-session flags, irrelevant for DB<br>
**'hashtag', 'chapters', 'volumes'** - near 100% null coverImage_large / _medium - redundant<br>
**title_userPreferred** - redundant with title_romaji<br>

In [175]:
DROP_COLS = [
    "Unnamed: 0",
    "type",
    "isFavourite",
    "isLocked",
    "hashtag",
    "chapters",
    "volumes",
    "coverImage_large",
    "coverImage_medium",
    "title_userPreferred",
]
df.drop(columns=DROP_COLS, inplace=True, errors="ignore")
print(f"  After drop: {len(df.columns)} columns")

  After drop: 52 columns


---
# 3. Fixing Data Types


In [176]:
#Integers
int_cols = [
    "idMal", "startDate_year", "startDate_month", "startDate_day",
    "endDate_year", "endDate_month", "endDate_day",
    "seasonYear", "seasonInt", "episodes", "duration",
    "averageScore", "meanScore", "popularity", "favourites", "trending",
]
for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

In [177]:
#Bool
bool_cols = ["isLicensed", "isAdult"]
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].map({True: True, False: False, "True": True, "False": False})

In [178]:
#TimeStamps to datetime
if "updatedAt" in df.columns:
    df["updatedAt"] = pd.to_datetime(df["updatedAt"], unit="s", errors="coerce")

In [179]:
def build_date(row, prefix):
    y = row.get(f"{prefix}_year")
    if pd.isna(y):
        return None
    m = row.get(f"{prefix}_month")
    d = row.get(f"{prefix}_day")
    return {
        "year":  int(y),
        "month": None if pd.isna(m) else int(m),
        "day":   None if pd.isna(d) else int(d),
    }

df["startDate"] = df.apply(lambda r: build_date(r, "startDate"), axis=1)
df["endDate"]   = df.apply(lambda r: build_date(r, "endDate"),   axis=1)

df.drop(columns=[
    "startDate_year","startDate_month","startDate_day",
    "endDate_year","endDate_month","endDate_day",
], inplace=True, errors="ignore")

In [180]:
#Titles
df["title"] = df.apply(lambda r: {
    "romaji":  r.get("title_romaji"),
    "english": r.get("title_english"),
    "native":  r.get("title_native"),
}, axis=1)
df.drop(columns=["title_romaji","title_english","title_native"], inplace=True, errors="ignore")

In [181]:
#CoverImages
df["coverImage"] = df.apply(lambda r: {
    "extraLarge": r.get("coverImage_extraLarge"),
    "color":      r.get("coverImage_color"),
}, axis=1)
df.drop(columns=["coverImage_extraLarge","coverImage_color"], inplace=True, errors="ignore")

In [182]:
#Trailers
df["trailer"] = df.apply(lambda r: {
    "id":        r.get("trailer_id"),
    "site":      r.get("trailer_site"),
    "thumbnail": r.get("trailer_thumbnail"),
} if pd.notna(r.get("trailer_id")) else None, axis=1)
df.drop(columns=["trailer_id","trailer_site","trailer_thumbnail"], inplace=True, errors="ignore")

In [183]:
print("  Data types fixed, nested fields reconstructed")

  Data types fixed, nested fields reconstructed


---
# 4. Basic Data Quality


In [184]:
before = len(df)
df.drop_duplicates(subset=["id"], inplace=True)
print(f"  Duplicates removed: {before - len(df)} rows")

# Drop rows with no meaningful identity (no id or no title)
df = df[df["id"].notna()]
print(f"  Rows after null-id drop: {len(df)}")

for score_col in ["averageScore", "meanScore"]:
    if score_col in df.columns:
        invalid = ((df[score_col] < 0) | (df[score_col] > 100)).sum()
        if invalid > 0:
            print(f"  WARNING: {invalid} out-of-range values in '{score_col}' — clamping")
            df[score_col] = df[score_col].clip(0, 100)

  Duplicates removed: 0 rows
  Rows after null-id drop: 20329


---
# 5. Parcing Part
> he-he pp

In [185]:
JSON_COLS = [
    "genres", "synonyms", "tags", "rankings", "externalLinks",
    "streamingEpisodes", "relations", "characters", "staff", "studios",
    "airingSchedule", "recommendations", "reviews",
    "stats_scoreDistribution", "stats_statusDistribution",
    "nextAiringEpisode",
]

def safe_parse(val):
    if isinstance(val, (list, dict)):
        return val
    if not isinstance(val, str):
        return None
    if val in ("", "[]", "{}"):
        return None
    try:
        return json.loads(val)
    except (json.JSONDecodeError, TypeError):
        return None

for col in JSON_COLS:
    if col in df.columns:
        df[col] = df[col].apply(safe_parse)

print("  JSON columns parsed")

  JSON columns parsed


---
# 6. Final Reports
> i love when lotta text spitting

In [186]:
print("\n=== Final shape ===")
print(f"Rows   : {len(df)}")
print(f"Columns: {len(df.columns)}")
print(f"Columns: {list(df.columns)}")

print("\n=== Null % per column ===")
nulls = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(nulls[nulls > 0].round(1).to_string())


=== Final shape ===
Rows   : 20329
Columns: 43
Columns: ['id', 'idMal', 'format', 'status', 'description', 'season', 'seasonYear', 'seasonInt', 'episodes', 'duration', 'countryOfOrigin', 'isLicensed', 'source', 'updatedAt', 'bannerImage', 'genres', 'synonyms', 'tags', 'averageScore', 'meanScore', 'popularity', 'favourites', 'trending', 'rankings', 'isAdult', 'siteUrl', 'externalLinks', 'streamingEpisodes', 'relations', 'characters', 'staff', 'studios', 'nextAiringEpisode', 'airingSchedule', 'recommendations', 'reviews', 'stats_scoreDistribution', 'stats_statusDistribution', 'startDate', 'endDate', 'title', 'coverImage', 'trailer']

=== Null % per column ===
nextAiringEpisode    99.9
streamingEpisodes    86.2
reviews              82.3
airingSchedule       76.1
bannerImage          63.8
trailer              63.1
rankings             56.5
externalLinks        46.2
recommendations      39.4
characters           36.1
synonyms             36.1
seasonYear           31.3
season               

---
# 7. Saving it as JSON
> Mongo-ready

In [ ]:
import numpy as np

print(f"\nSaving to {OUTPUT_PATH}...")

def convert(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

records = df.where(pd.notna(df), None).to_dict(orient="records")

# pandas preserves np.nan in numeric columns even after .where() — sanitize explicitly
records = [
    {k: (None if isinstance(v, (float, np.floating)) and not np.isfinite(v) else v)
     for k, v in r.items()}
    for r in records
]

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, default=convert)

print(f"Done. {len(records)} documents saved to '{OUTPUT_PATH}'")
#  mongoimport --db anilist_db --collection anime --file Data/anilist_cleaned.json --jsonArray
